(sec_PythonIntegration)=
# Integration in `Python`

This section is intended to be a compendium (hopefully clear and well organized) of all the `Python` we have used in Chapter 4.

Objectives:

* Computing antiderivatives with `Sympy`.
* Computing definite integrals with `Sympy`.
* Implementing numerical integration methods in `Numpy`.
* Computing improper integrals with `Sympy`.
* Using `Sympy` to solve ODEs.


## Computing antiderivatives with `SymPy`

To compute an antiderivative with `SymPy`, we use the *integrate* function. For example, to find an antiderivative of $\sin(x)$, we write:


In [86]:
import sympy as sp

x = sp.symbols('x')
f_exp = sp.sin(x)
I = sp.integrate(f_exp,x)

print('An antiderivative of ',f_exp, ' is = ',I)

An antiderivative of  sin(x)  is =  -cos(x)


`SymPy` cannot always compute an antiderivative. When it cannot, it returns the original integral as its output:


In [87]:
I = sp.integrate(sp.sin(x*sp.cos(x)),x)
print(I)

Integral(sin(x*cos(x)), x)


## Computing definite integrals with `Sympy`

To compute a definite integral, we simply add the integration limits to the `sp.integrate` command.

For example, to compute $\displaystyle\int_0^\pi\sin(x)\,dx$, we write:


In [88]:
import sympy as sp

x = sp.symbols('x')
f_exp = sp.sin(x)
Idef = sp.integrate(f_exp,(x,0,sp.pi))  # Definite integral of f_exp over x from 0 to pi

print('The integral of ',f_exp, ' between 0 and pi is = ',Idef)

The integral of  sin(x)  between 0 and pi is =  2


## Numerical integration with `Numpy`

### Simple formulas

Below are functions that implement the simple formulas we have just studied in `Numpy`, followed by an example of their application. We will apply them to

$$
I=\int_{0}^{3}\left(x^4+1\right)\,dx\,,
$$
since in this simple case we can find the exact value of the integral:

$$
I=\int_{0}^{3}\left(x^4+1\right)\,dx = \left[\frac{x^5}{5}+x\right]_{x=0}^{3} = \frac{3^5}{5}+3 = 51.6\,.
$$


In [89]:
import sympy as sp
import numpy as np

def midpoint_rule(a, b, fpm):
    approx_midpoint = (b-a) * fpm
    return approx_midpoint

def trapezoidal_rule(a, b, fa, fb):
    approx_trapezoid = (b-a) * (fa + fb)/2
    return approx_trapezoid

def simpson_rule(a, b, fa, fpm, fb):
    approx_simpson = (b-a) * (fa + 4*fpm + fb)/6
    return approx_simpson

x = sp.Symbol('x', real = True)

f_exp = x**4 + 1
f = sp.lambdify(x,f_exp)

a = 0
b = 3
pm = (a+b)/2

fa = f(a)
fpm = f(pm)
fb = f(b)

print('Approximate value of I using the midpoint formula = ', midpoint_rule(a,b,fpm) ) 
print('Approximate value of I using the trapezoidal rule = ', trapezoidal_rule(a,b,fa,fb) ) 
print("Approximate value of I using Simpson's rule = ", simpson_rule(a,b,fa,fpm,fb) ) 

Approximate value of I using the midpoint formula =  18.1875
Approximate value of I using the trapezoidal rule =  124.5
Approximate value of I using Simpson's rule =  53.625


### Composite formulas

As you can see in the previous section, simple formulas can be quite... poor.

We will now efficiently implement the composite formulas using the `np.sum` function.


In [90]:
import sympy as sp
import numpy as np

x = sp.Symbol('x', real = True)
f_exp = x**4+1
f = sp.lambdify(x,f_exp)

a = 0; b = 3
n = 100

x1 = np.linspace(a,b,n+1) # here we store the x_{i}. 
                          # Remember that Python stores x1[0], x1[1], ..., x1[(n+1)-1] = x1[n]
y1 = f(x1)

h = (b-a)/n # the width of each subinterval

approx_trapezoid = h/2 * (y1[0]+2*np.sum(y1[1:n])+y1[n])
approx_midpoint = 2*h * np.sum(y1[1:n:2])
approx_simpson = 2*h/6 * (y1[0] + 4*np.sum(y1[1:n:2])+2*np.sum(y1[2:n-1:2])+y1[n])

print('approx_trapezoid: ',approx_trapezoid) 
print('approx_midpoint: ',approx_midpoint) 
print('approx_simpson: ',approx_simpson) 

print('Exact: ',b**5/5+b)

approx_trapezoid:  51.608099919
approx_midpoint:  51.583801133999984
approx_simpson:  51.60000032399999
Exact:  51.6


## Computing improper integrals with `Sympy`

`Sympy` can compute improper integrals of the first kind, that is, integrals whose limits of integration are $-\infty$ and/or $+\infty$.

This can be done either directly or by applying the definition of an improper integral (that is, combining a Riemann integral with a limit). Let us see how:


In [91]:
import sympy as sp
x = sp.symbols('x', real = True)
M = sp.Symbol('M', real = True)

f_exp = sp.exp(x)

# Direct computation
I_direct = sp.integrate(f_exp,(x,-sp.oo,0))
print('Integral of ',f_exp,' between -oo and 0 is = ',I_direct)

# Computation using limits
I_limits = sp.limit( sp.integrate(f_exp,(x,-M,0)), M, +sp.oo )
print('Integral of ',f_exp,' between -oo and 0 is = ',I_limits)

Integral of  exp(x)  between -oo and 0 is =  1
Integral of  exp(x)  between -oo and 0 is =  1


We can also compute an improper integral of the second kind. For example,

$$
\int_{0}^2\dfrac{1}{\sqrt{x}}\,dx\,.
$$


In [92]:
import sympy as sp
x = sp.symbols('x', real = True)
c = sp.Symbol('c', real = True)

f_exp = 1/sp.sqrt(x)

# Direct computation
I_direct = sp.integrate(f_exp,(x,0,2))
print('The integral equals = ', I_direct)

# Computation using limits
I_limits = sp.limit( sp.integrate(f_exp,(x,c,2)), c, 0, dir='+')
print('The integral equals = ', I_limits)

The integral equals =  2*sqrt(2)
The integral equals =  2*sqrt(2)


Of course, we may also encounter divergent integrals:


In [93]:
import sympy as sp
x = sp.symbols('x', real = True)
c = sp.Symbol('c', real = True)

f_exp = 1/x

# Computation using limits
I_limits = sp.limit( sp.integrate(f_exp,(x,c,2)), c, 0, dir='+')
print('The integral equals = ', I_limits)

The integral equals =  oo


## Using `Sympy` to solve ODEs

Below we show how to use `Sympy` to solve ODEs.

The process is straightforward:

1. Define independent variables as symbols (`sp.Symbol`) and dependent variables as functions (`sp.Function`).
2. Define the ODE with the `sp.Eq` command, emphasizing the dependence of the dependent variable on the independent variable. In the example below, we write `v(x)` wherever the dependent variable $v$ appears.
3. Write derivatives inside `sp.Eq` by specifying the dependent variable and the variable with respect to which it is differentiated. In the example below, we write $v\prime$ as `v(x).diff(x)`.
4. Once the ODE is defined, solve it with the `sp.dsolve` command.
5. We can call `sp.dsolve` without additional arguments to find the general solution, or include an initial condition, defined with `ics`, as in the final line of the code below.

As an example, let us compute the velocity of a body with mass $72$ kilograms, assuming its initial velocity is zero and its air resistance coefficient is $k=0.2$.

That is, using the ideas from the previous section, we will solve the initial value problem

$$
    \left\{\begin{array}{rcl}
    72 v\prime&=& 72\cdot 9.81 - 0.2 v\,,\\
    v(0) &=& 0\,,
    \end{array}\right.
$$


In [94]:
import sympy as sp

# Independent variable
x = sp.Symbol ('x')
# Dependent variable (defined as a Function)
v = sp.Function ('v')

# Define the ODE 
eq = sp.Eq (72*v(x).diff(x), 72*9.81 - 0.2*v(x))

# Compute the general solution (this step is not necessary, but is included as an example)
general_solution = sp.dsolve (eq)   
display (general_solution)

# Compute the requested particular solution
particular_solution = sp.dsolve (eq, ics={v(0): 0.0}) 
display (particular_solution)

Eq(v(x), C1*exp(-0.00277777777777778*x) + 3531.6)

Eq(v(x), 3531.6 - 3531.6*exp(-0.00277777777777778*x))

---
Adapted and translated from [the Spanish-language notebook *Integración en Python*](https://github.com/GEI-CAL/JB-Calculo1-UDC/blob/main/capitulos/05/06.Integracion.ipynb) by GEI-CAL. The original is licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); this English translation is an adaptation.
